# 03 · Metrica sample matches: labels from events

The three public Metrica Sports sample matches have real tracking dropouts and are closer to the challenge during stoppages than the degraded DFL data. They are therefore used **without degradation** as additional training matches. They have no frame labels; we derive them automatically from the Metrica events.

**Output:** `artifacts/metrica/MET{1,2,3}.parquet`, training data for notebook 04.

In [1]:
# Standard library
import json
from pathlib import Path

# Data
import numpy as np
import pandas as pd

# Tracking loader
from kloppy import metrica

ROOT = Path.cwd().resolve().parent
MET_RAW = ROOT / "external_data" / "metrica_sample" / "raw"
MET_OUT = ROOT / "artifacts" / "metrica"
MET_OUT.mkdir(parents=True, exist_ok=True)

FPS = 25
PITCH_MET = np.array([105.0, 68.0])  # metadata of the sample matches
MAX_BALL_SPEED, MAX_PLAYER_SPEED = 40.0, 12.0  # as in notebook 02
POSS_TYPES = {"PASS", "CARRY", "RECOVERY", "SET PIECE", "SHOT", "BALL LOST", "BALL OUT"}
STOP_LOST = {"OFFSIDE", "HAND BALL", "END HALF"}

## 1. Read tracking and events

Matches 1 and 2 are CSV files, match 3 uses the EPTS format. Coordinates as for the challenge: `(v − 0.5) · pitch size`.

In [2]:
def clean(ball, players):
    ball, players = ball.copy(), players.copy()
    ball[np.r_[0, np.linalg.norm(np.diff(ball, axis=0), axis=1) * FPS] > MAX_BALL_SPEED] = np.nan
    step = np.linalg.norm(np.diff(players, axis=0), axis=-1) * FPS
    players[np.r_[np.zeros((1, players.shape[1])), step] > MAX_PLAYER_SPEED] = np.nan
    return ball, players

def tracking(g):
    if g == 3:
        ds = metrica.load_tracking_epts(meta_data=MET_RAW / "Sample_Game_3_metadata.xml", raw_data=MET_RAW / "Sample_Game_3_tracking.txt")
    else:
        ds = metrica.load_tracking_csv(home_data=MET_RAW / f"Sample_Game_{g}_RawTrackingData_Home_Team.csv",
                                       away_data=MET_RAW / f"Sample_Game_{g}_RawTrackingData_Away_Team.csv")
    df = ds.to_df(engine="pandas")
    ids = [(s, p.player_id) for s, t in zip("HA", ds.metadata.teams) for p in t.players if f"{p.player_id}_x" in df]
    xy = lambda cols: (np.stack([df[[f"{c}_{a}" for c in cols]].to_numpy(float) for a in "xy"], -1) - 0.5) * PITCH_MET
    return df["frame_id"].to_numpy(), df["period_id"].to_numpy(), xy(["ball"])[:, 0], xy([i for _, i in ids]), np.array([s for s, _ in ids]), ds.metadata.teams[0].name

def events(g, home):
    if g == 3:
        sub = lambda s: "" if not s else s["name"] if isinstance(s, dict) else "-".join(x["name"] for x in s)
        e = pd.DataFrame([{"team": "H" if x["team"]["name"] == home else "A", "type": x["type"]["name"], "sub": sub(x["subtypes"]),
                           "period": x["period"], "start": x["start"]["frame"], "end": x["end"]["frame"]}
                          for x in json.load(open(MET_RAW / "Sample_Game_3_events.json"))["data"]])
    else:
        r = pd.read_csv(MET_RAW / f"Sample_Game_{g}_RawEventsData.csv")
        e = pd.DataFrame({"team": r["Team"].map({"Home": "H", "Away": "A"}), "type": r["Type"], "sub": r["Subtype"].fillna(""),
                          "period": r["Period"], "start": r["Start Frame"], "end": r["End Frame"]})
    e["end"] = np.maximum(e["start"], e["end"])  # kick-offs have End Frame 0 in matches 1/2
    return e.sort_values("start", kind="stable").reset_index(drop=True)

raw = {}
for g in (1, 2, 3):
    frame, period, ball, players, side, home = tracking(g)
    raw[g] = {"frame": frame, "period": period, "ball": ball, "players": players, "side": side, "events": events(g, home)}
    print(f"Match {g}: {len(frame):,} frames · {len(side)} players · {len(raw[g]['events']):,} events · ball missing in {np.isnan(ball[:, 0]).mean():.1%}")

Match 1: 145,006 frames · 28 players · 1,745 events · ball missing in 39.1%


Match 2: 141,156 frames · 26 players · 1,935 events · ball missing in 41.0%


Match 3: 143,761 frames · 35 players · 3,620 events · ball missing in 32.2%


## 2. Label rule

- **`D` starts** at the end of `BALL OUT`, at a shot going out or into the goal, at a foul (`CHALLENGE …FAULT…`, `FAULT RECEIVED`) and at offside, handball or end of half.
- **`D` ends** at the next `SET PIECE` or goal kick.
- **In play**, the ball belongs to the team of the last possession event (pass, carry, recovery, set piece, shot, ball lost). Challenges, cards and fouls do not transfer possession.

Each half is labelled from the first set piece (kick-off) to the last event. Then the same plausibility rule as in notebook 02 is applied.

In [3]:
def points(e):
    tok = e["sub"].str.split("-").apply(set)
    stop_end = (e["type"] == "BALL OUT") | ((e["type"] == "SHOT") & tok.apply(lambda t: bool(t & {"OUT", "GOAL"}))) | \
               ((e["type"] == "BALL LOST") & tok.apply(lambda t: bool(t & STOP_LOST)))
    stop_start = ((e["type"] == "CHALLENGE") & tok.apply(lambda t: "FAULT" in t)) | (e["type"] == "FAULT RECEIVED")
    restart = (e["type"] == "SET PIECE") | (e["type"].isin(["PASS", "BALL LOST", "BALL OUT"]) & tok.apply(lambda t: "GOAL KICK" in t))
    p = pd.concat([pd.DataFrame({"frame": e.loc[stop_end, "end"], "kind": 0}), pd.DataFrame({"frame": e.loc[stop_start, "start"], "kind": 0}),
                   pd.DataFrame({"frame": e.loc[restart, "start"], "kind": 1})])
    return p.sort_values(["frame", "kind"]).to_numpy()

def label_game(r):
    e, frame = r["events"], r["frame"]
    keep, lab = np.zeros(len(frame), bool), np.full(len(frame), "", dtype="<U1")
    for per in (1, 2):
        ep = e[e["period"] == per]
        first = ep.loc[(ep["type"] == "SET PIECE"), "start"].min()
        inside = (r["period"] == per) & (frame >= first) & (frame <= ep["end"].max())
        keep |= inside

        poss = ep[ep["type"].isin(POSS_TYPES)]
        j = np.searchsorted(poss["start"].to_numpy(), frame[inside], side="right") - 1
        lab[inside] = poss["team"].to_numpy()[np.maximum(j, 0)]

        dead, open_at = np.zeros(len(frame), bool), None
        for f, kind in points(ep):
            if kind == 0 and open_at is None:
                open_at = f
            elif kind == 1 and open_at is not None:
                dead |= (frame >= open_at) & (frame < f)
                open_at = None
        if open_at is not None:
            dead |= frame >= open_at
        lab[inside & dead] = "D"
    return keep, lab

summary = {}
for g, r in raw.items():
    keep, lab = label_game(r)
    ball, players = r["ball"][keep], r["players"][keep]
    per = r["period"][keep]
    cb, cp = zip(*[clean(ball[per == p], players[per == p]) for p in (1, 2)])
    ball, players = np.concatenate(cb), np.concatenate(cp)

    cols = {"frame": r["frame"][keep], "period": per, "label": lab[keep], "ball_x": ball[:, 0], "ball_y": ball[:, 1]}
    for k, s in enumerate(r["side"]):
        cols[f"{s}_P{k:02d}_x"], cols[f"{s}_P{k:02d}_y"] = players[:, k, 0], players[:, k, 1]
    pd.DataFrame(cols).to_parquet(MET_OUT / f"MET{g}.parquet", index=False)
    summary[f"MET{g}"] = {"minutes": keep.sum() / FPS / 60, **pd.Series(lab[keep]).value_counts(normalize=True).to_dict()}

pd.DataFrame(summary).T.round(3)

,minutes,D,H,A
MET1,94.716,0.449,0.278,0.273
MET2,93.409,0.415,0.303,0.282
MET3,95.132,0.298,0.366,0.337


**Result:** three matches with automatically derived H/A/D labels. The label rule is based on the Metrica events, not on the challenge. The event timestamps were recorded by humans and are not frame-exact.